In [ ]:
import sys
from pathlib import Path

_root = next(p for p in Path.cwd().resolve().parents if (p / '.git').exists() or (p / 'setup.py').exists())
for _p in (str(_root), str(_root / 'src')):
    if _p not in sys.path:
        sys.path.insert(0, _p)

import numpy as np
import scanpy as sc
import gseapy as gp
from metab_processing.metab_travlr_config import DATA_DIR
from metab_processing.SpaceTravLR.dataset_configs import dataset_paths
from metab_processing.LinearRegression.build_x import add_gene_signature
from metab_processing.LinearRegression.least_squares import (
    fit_gene_betas, subsample_gene_betas, rank_coefficients,
    plot_top_coefficients, plot_beta_histogram)

dataset = 'Primary_Dermal_Melanoma'
data_dir = f'{DATA_DIR}/Xenium'
ANNOT_COL, CELL_TYPE = 'Tier1', 'T Cell'
METHOD, PENALTY, STANDARDIZE = 'l1', 0.001, True
N_SUB = 200
LOWER, UPPER = 2.5, 97.5
DBVER = '2024.1.Hs'

SIGNATURES = [
    # T cell biology
    'HALLMARK_IL2_STAT5_SIGNALING',
    'KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY',
    'GOBP_T_CELL_MEDIATED_CYTOTOXICITY',
    'KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY',
    'HALLMARK_TNFA_SIGNALING_VIA_NFKB',
    'HALLMARK_INTERFERON_GAMMA_RESPONSE',
    # Metabolic
    'HALLMARK_OXIDATIVE_PHOSPHORYLATION',
    'HALLMARK_GLYCOLYSIS',
    'HALLMARK_HYPOXIA',
    'HALLMARK_FATTY_ACID_METABOLISM',
    'HALLMARK_MTORC1_SIGNALING',
    # Negative control
    'HALLMARK_PANCREAS_BETA_CELLS',
]

In [ ]:
paths = dataset_paths(dataset, data_dir=data_dir)
adata = sc.read_h5ad(paths['dataset_dir'] / 'LinearRegression' / 'x_adata.h5ad')
var = set(adata.var_names)
print(adata.n_obs, 'cells |', len(adata.uns['x_genes']), 'built genes')

In [ ]:
# Fetch the gene sets, print each, and report which genes are absent from the data.
msig = gp.Msigdb()
library = {}
for cat in ('h.all', 'c2.cp.kegg_legacy', 'c5.go.bp'):
    try:
        library.update(msig.get_gmt(category=cat, dbver=DBVER) or {})
    except Exception as e:
        print(f'{cat}: FAILED ({e})')

sets = {}        # name -> genes PRESENT in adata (what each pseudo-gene is built from)
for name in SIGNATURES:
    genes = library.get(name)
    if genes is None:
        print(f'MISSING SET (name not found in MSigDB): {name}')
        continue
    present = [g for g in genes if g in var]
    absent = [g for g in genes if g not in var]
    sets[name] = present
    print(f'{name}: {len(present)}/{len(genes)} genes present'
          + (f'  | missing: {absent}' if absent else ''))

In [ ]:
def add_subsample_percentiles(betas_df, sub, lower=LOWER, upper=UPPER):
    """Add min/max and lower/upper-percentile subsample-beta columns to betas_df,
    matched by factor (`sub` = n_subsamples x factor DataFrame from subsample_gene_betas)."""
    out = betas_df.copy()
    def stat(fn):
        return [fn(sub[f]) if f in sub.columns else np.nan for f in out['factor']]
    out['min_subsample_beta'] = stat(np.nanmin)
    out['max_subsample_beta'] = stat(np.nanmax)
    out[f'lower_{lower}_subsample_beta'] = stat(lambda s: np.nanpercentile(s, lower))
    out[f'upper_{upper}_subsample_beta'] = stat(lambda s: np.nanpercentile(s, upper))
    return out

In [ ]:
def run_signature(name, penalty=PENALTY, n_sub=N_SUB, lower=LOWER, upper=UPPER, top=15):
    """All-positive pseudo-gene for `name` -> fit (L1/standardized) -> resample -> ranked
    betas with subsample-percentile columns. Prints a table + a top-coefficients plot.
    Returns (betas_with_percentiles, sub)."""
    if name not in sets:
        print(f'skip {name}: set not available'); return None, None
    genes = sets[name]
    ad2 = add_gene_signature(adata, name, positive=genes)   # negative=[] (all positive)
    b = fit_gene_betas(ad2, genes=[name], annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                       method=METHOD, penalty=penalty, standardize=STANDARDIZE)
    sub = subsample_gene_betas(ad2, name, factors=None, n_subsamples=n_sub,
                               annot_col=ANNOT_COL, annot_value=CELL_TYPE,
                               method=METHOD, penalty=penalty, standardize=STANDARDIZE)
    res = add_subsample_percentiles(rank_coefficients(b), sub, lower, upper)
    print(f'=== {name}: {len(genes)} genes present ===')
    display(res.head(20))
    if not res.empty:
        plot_top_coefficients(b, top=top).set_title(name)
    return res, sub

## T cell biology

In [ ]:
res_il2, sub_il2 = run_signature('HALLMARK_IL2_STAT5_SIGNALING')

In [ ]:
res_tcr, sub_tcr = run_signature('KEGG_T_CELL_RECEPTOR_SIGNALING_PATHWAY')

In [ ]:
res_tcyto, sub_tcyto = run_signature('GOBP_T_CELL_MEDIATED_CYTOTOXICITY')

In [ ]:
res_nk, sub_nk = run_signature('KEGG_NATURAL_KILLER_CELL_MEDIATED_CYTOTOXICITY')

In [ ]:
res_tnfa, sub_tnfa = run_signature('HALLMARK_TNFA_SIGNALING_VIA_NFKB')

In [ ]:
res_ifng, sub_ifng = run_signature('HALLMARK_INTERFERON_GAMMA_RESPONSE')   # positive control

## Metabolic

In [ ]:
res_oxphos, sub_oxphos = run_signature('HALLMARK_OXIDATIVE_PHOSPHORYLATION')

In [ ]:
res_glyc, sub_glyc = run_signature('HALLMARK_GLYCOLYSIS')

In [ ]:
res_hyp, sub_hyp = run_signature('HALLMARK_HYPOXIA')

In [ ]:
res_fao, sub_fao = run_signature('HALLMARK_FATTY_ACID_METABOLISM')

In [ ]:
res_mtor, sub_mtor = run_signature('HALLMARK_MTORC1_SIGNALING')

## Negative control

In [ ]:
res_neg, sub_neg = run_signature('HALLMARK_PANCREAS_BETA_CELLS')

## Distribution of a single factor (edit the signature / factor)

In [ ]:
sub = sub_glyc                 # <- any signature's resample output from above
FACTOR = sub.columns[0]        # <- any factor name (tf / lig$rec / lig#tf / metab@name)
plot_beta_histogram(sub[FACTOR], title=FACTOR);